# Build the Modeling Table — Phases 0-2

Implements **Build Sequence Phases 0, 1, and 2** from the project spec
(`README.md` §3, §9):

| Phase | Task | Output |
|---|---|---|
| **0** | Sort all 59 `cps_filers_clean.csv` columns into piles (§3.4) | `data_dictionary.md` |
| **1** | Filter to filer units (`depstat==0`, §3.2), confirm `eff_rate` target, build income-share features | clean filer table |
| **2** | **Freeze** the modeling table (80/20, fixed seed) — *blocks all downstream work* | `train.csv`, `test.csv` |

**Decisions carried from the spec (with the recommended defaults):**
- Filer unit = **(A) `depstat == 0`** (non-dependent filers) — §3.2, recommended.
- Target = **`eff_rate`** used directly (already `100 × fedtaxac / adjginc`) — §3.3.
- Negative effective rates (refundable-credit filers) are **kept, not clipped** — §3.3.
- Feature scope = **core only** — income level + income-composition shares +
  filing/marital/household/age/geography. Optional context axes and all
  sensitive axes (race, ethnicity, immigration, sex, …) are **excluded** from
  `X`, per the locked income / filing-status / geography framing (§3.4, §10.4).
- Split = **80/20, `random_state=42`** — §3.5.


## Setup

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

RANDOM_STATE = 42  # fixed seed for every stochastic step (reproducibility, §4.1)

# Resolve the repo root whether this notebook runs from notebooks/ or the root.
ROOT = Path.cwd()
while not (ROOT / "README.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

RAW_CSV   = ROOT / "data" / "raw" / "cps_filers_clean.csv"
LABELS    = ROOT / "data" / "raw" / "data_dictionary.csv"   # variable labels + value codes
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_csv(RAW_CSV)
labels = pd.read_csv(LABELS).set_index("variable")["label"].to_dict()

print("Loaded:", RAW_CSV.relative_to(ROOT))
print("Shape :", df_raw.shape, "(spec expects 64,696 x 59)")
assert df_raw.shape == (64696, 59), "input file does not match the spec's 64,696 x 59"


Loaded: data/raw/cps_filers_clean.csv
Shape : (64696, 59) (spec expects 64,696 x 59)


## Phase 0 — Column pile-sort (leakage firewall)

Every one of the 59 columns is assigned to **exactly one** pile. Misassignment
causes data leakage — a model that peeks at the answer (§3.4, §10.1).
**When timing is ambiguous, quarantine.**

Pile vocabulary:

| Pile | Meaning |
|---|---|
| `target` | The value we predict (`eff_rate`). |
| `feature` | Pre-tax, known before any tax is computed → enters `X`. |
| `optional_feature` | Legitimate pre-tax axis, but **out of scope** for the locked framing → documented, not in `X`. |
| `quarantine` | Target components or post-tax/credit outputs → **never** a feature (leakage). |
| `weight` | Survey weights — used for weighted validation, **never** in `X`. |
| `sensitive` | Race / ethnicity / immigration / sex / disability — excluded from the equity framing (§10.4). |
| `drop` | Identifiers, administrative fields, redundant/household-level columns, and the filter selector. |


In [2]:
# Authoritative pile assignment for all 59 columns: col -> (pile, rationale).
# This dict is the single source of truth; data_dictionary.md is generated from it.
COLUMN_PILES = {
    # --- target -------------------------------------------------------------
    "eff_rate":   ("target", "Prediction target: 100 x fedtaxac / adjginc (percent)."),

    # --- features (pre-tax; core framing) -----------------------------------
    "inctot":     ("feature", "Pre-tax total personal income; income level + denominator for shares."),
    "incwage":    ("feature", "Source of wage_share (income composition; core equity signal)."),
    "incbus":     ("feature", "Source of business_share; can be negative (losses)."),
    "incint":     ("feature", "Source of interest_share."),
    "incdivid":   ("feature", "Source of dividend_share; preferentially taxed capital income."),
    "incretir":   ("feature", "Source of retirement_share; nulls imputed to 0."),
    "incss":      ("feature", "Source of socsec_share; partially/untaxed."),
    "incrent":    ("feature", "Source of rent_share; can be negative (losses)."),
    "filestat":   ("feature", "Filing status. Legitimate feature but CPS-tax-model assigned -> flagged leakage risk (§3.4)."),
    "marst":      ("feature", "Marital status; structural driver independent of imputed filestat."),
    "nchild":     ("feature", "Number of own children; drives dependent credits/deductions."),
    "nchlt5":     ("feature", "Young children (<5); CTC / childcare relevance."),
    "famsize":    ("feature", "Family size; household scale."),
    "age":        ("feature", "Age; retirement / senior treatment."),
    "statefip":   ("feature", "State (FIPS); geographic axis. Categorical (51 levels) -> one-hot at model time."),

    # --- optional features (out of scope for the locked framing) ------------
    "educ":       ("optional_feature", "Education; context axis, not in the core income/filing/geography framing."),
    "empstat":    ("optional_feature", "Employment status; optional context axis."),
    "wkswork1":   ("optional_feature", "Weeks worked last year; optional context axis."),
    "uhrsworkly": ("optional_feature", "Usual hours/week; optional context axis."),
    "classwkr":   ("optional_feature", "Class of worker (self-employed vs wage); optional context axis."),
    "pension":    ("optional_feature", "Pension coverage; optional context axis."),
    "firmsize":   ("optional_feature", "Firm size; optional context axis."),

    # --- quarantine (target components / post-tax / credits) ----------------
    "fedtaxac":   ("quarantine", "Numerator of the target (federal tax after credits)."),
    "adjginc":    ("quarantine", "Denominator of the target (AGI)."),
    "spmfedtaxac":("quarantine", "SPM-unit federal tax; same outcome, different grouping. Leakage."),
    "eitcred":    ("quarantine", "Refundable credit; computed as part of the tax outcome."),
    "ctccrd":     ("quarantine", "Child Tax Credit; post-tax credit."),
    "actccrd":    ("quarantine", "Additional (refundable) CTC; post-tax credit."),
    "margtax":    ("quarantine", "Marginal tax rate; direct function of the tax computation."),
    "taxinc":     ("quarantine", "Taxable income; post-deduction, downstream of AGI."),
    "fica":       ("quarantine", "Payroll tax; a computed tax outcome (kept out to keep 'pre-tax' clean)."),

    # --- survey weights (validation only) -----------------------------------
    "asecwt":     ("weight", "ASEC person weight; population-representative validation. Never in X."),
    "asecwth":    ("weight", "ASEC household weight; validation weight. Never in X."),

    # --- sensitive (excluded from the equity framing, §10.4) ----------------
    "sex":        ("sensitive", "Sensitive; include only if the equity framing explicitly covers it (it does not)."),
    "race":       ("sensitive", "Race; not an agreed equity axis. Never a causal driver in this project."),
    "hispan":     ("sensitive", "Hispanic origin; not an agreed equity axis."),
    "citizen":    ("sensitive", "Citizenship; immigration axis, out of framing."),
    "nativity":   ("sensitive", "Nativity; immigration axis, out of framing."),
    "yrimmig":    ("sensitive", "Year of immigration; immigration axis, out of framing."),
    "vetstat":    ("sensitive", "Veteran status; sensitive, out of framing."),
    "diffany":    ("sensitive", "Disability; sensitive, out of framing."),

    # --- drop (ids, admin, redundant, filter selector) ----------------------
    "year":       ("drop", "Constant survey year; identifier/admin."),
    "serial":     ("drop", "Household serial number; identifier."),
    "month":      ("drop", "Survey month; admin."),
    "cpsid":      ("drop", "CPSID household record; identifier."),
    "cpsidp":     ("drop", "CPSID person record; identifier."),
    "cpsidv":     ("drop", "Validated longitudinal identifier; identifier."),
    "asecflag":   ("drop", "ASEC flag; admin."),
    "pernum":     ("drop", "Person number; identifier."),
    "ftype":      ("drop", "Family type; administrative grouping."),
    "spmfamunit": ("drop", "SPM unit id; grouping identifier."),
    "spmwt":      ("drop", "SPM unit weight; not used (person-level analysis)."),
    "spmftotval": ("drop", "SPM unit cash income; household-level, redundant/leakage risk."),
    "spmftotval_dup": None,  # placeholder removed below if unused
    "ftotval":    ("drop", "Total family income; household-level, redundant/leakage risk."),
    "hhincome":   ("drop", "Total household income; household-level, redundant with filer income + leakage risk."),
    "occ2010":    ("drop", "Occupation (479 levels); not in the §3.4 feature set."),
    "ind":        ("drop", "Industry; not in the §3.4 feature set."),
    "depstat":    ("drop", "Dependency-status selector for the filer-unit filter; constant (0) after filtering."),
}
COLUMN_PILES.pop("spmftotval_dup", None)

VALID_PILES = {"target", "feature", "optional_feature", "quarantine", "weight", "sensitive", "drop"}

# Integrity checks: every column classified exactly once, nothing invented.
assert set(COLUMN_PILES) == set(df_raw.columns), (
    "mismatch: missing=%s extra=%s"
    % (set(df_raw.columns) - set(COLUMN_PILES), set(COLUMN_PILES) - set(df_raw.columns))
)
assert len(COLUMN_PILES) == 59, len(COLUMN_PILES)
assert all(p in VALID_PILES for p, _ in COLUMN_PILES.values())

pile_of = {c: p for c, (p, _) in COLUMN_PILES.items()}
counts = pd.Series(pile_of).value_counts()
print("All 59 columns classified exactly once. Pile counts:")
print(counts.to_string())


All 59 columns classified exactly once. Pile counts:
drop                17
feature             15
quarantine           9
sensitive            8
optional_feature     7
weight               2
target               1


In [3]:
# Generate data_dictionary.md from COLUMN_PILES (Phase 0 deliverable).
ORDER = ["target", "feature", "optional_feature", "quarantine", "weight", "sensitive", "drop"]
PILE_DESC = {
    "target":           "The value we predict.",
    "feature":          "Pre-tax; enters the model matrix X.",
    "optional_feature": "Legitimate pre-tax axis, out of scope for the locked framing; documented, not in X.",
    "quarantine":       "Target components or post-tax/credit outputs; never a feature (leakage).",
    "weight":           "Survey weight; weighted validation only, never in X.",
    "sensitive":        "Sensitive axis excluded from the equity framing (§10.4).",
    "drop":             "Identifier, admin, redundant/household-level, or the filter selector.",
}

lines = []
lines.append("# Data Dictionary & Column Pile-Sort")
lines.append("")
lines.append("**Source:** `data/raw/cps_filers_clean.csv` — 64,696 rows x 59 columns "
             "(IPUMS CPS ASEC 2024, income year 2023), one row per person.")
lines.append("")
lines.append("> Auto-generated by `notebooks/build_modeling_table.ipynb` (Phase 0). "
             "Do not edit by hand — edit `COLUMN_PILES` in the notebook and re-run.")
lines.append("")
lines.append("This is the **leakage firewall** required by the spec (README §3.4, §9 Phase 0): "
             "every column is assigned to exactly one pile. When column timing is ambiguous, it is quarantined.")
lines.append("")

# summary table
lines.append("## Pile summary")
lines.append("")
lines.append("| Pile | Count | Meaning |")
lines.append("|---|---:|---|")
for p in ORDER:
    lines.append(f"| `{p}` | {int(counts.get(p, 0))} | {PILE_DESC[p]} |")
lines.append(f"| **total** | **{len(COLUMN_PILES)}** | |")
lines.append("")

# per-pile detail tables (preserve dict order within each pile)
for p in ORDER:
    cols = [c for c in COLUMN_PILES if pile_of[c] == p]
    if not cols:
        continue
    lines.append(f"## `{p}` ({len(cols)})")
    lines.append("")
    lines.append("| Column | Label | Rationale |")
    lines.append("|---|---|---|")
    for c in cols:
        lines.append(f"| `{c}` | {labels.get(c, '')} | {COLUMN_PILES[c][1]} |")
    lines.append("")

# engineered feature set (forward reference to Phase 1)
lines.append("## Modeling feature set `X` (built in Phase 1)")
lines.append("")
lines.append("Only `feature`-pile columns are used. Raw income components are transformed into "
             "**income-composition shares** (the core equity signal); the raw amounts do not enter `X`.")
lines.append("")
lines.append("| Feature | Type | Derivation |")
lines.append("|---|---|---|")
lines.append("| `inctot` | numeric | Pre-tax total personal income (income level). |")
lines.append("| `wage_share` | numeric | `incwage / inctot`, 0 when `inctot <= 0`. |")
lines.append("| `business_share` | numeric | `incbus / inctot`, 0 when `inctot <= 0`. |")
lines.append("| `interest_share` | numeric | `incint / inctot`, 0 when `inctot <= 0`. |")
lines.append("| `dividend_share` | numeric | `incdivid / inctot`, 0 when `inctot <= 0`. |")
lines.append("| `retirement_share` | numeric | `incretir(->0) / inctot`, 0 when `inctot <= 0`. |")
lines.append("| `socsec_share` | numeric | `incss / inctot`, 0 when `inctot <= 0`. |")
lines.append("| `rent_share` | numeric | `incrent / inctot`, 0 when `inctot <= 0`. |")
lines.append("| `filestat` | categorical | Filing status (flagged leakage risk, §3.4). |")
lines.append("| `marst` | categorical | Marital status. |")
lines.append("| `statefip` | categorical | State FIPS (51 levels) -> one-hot at model time. |")
lines.append("| `nchild` | numeric | Number of own children. |")
lines.append("| `nchlt5` | numeric | Own children under 5. |")
lines.append("| `famsize` | numeric | Family size. |")
lines.append("| `age` | numeric | Age. |")
lines.append("")
lines.append("**Target:** `eff_rate` (percent). **Retained non-feature column:** `asecwt` (weighted validation, §3.5).")
lines.append("")
lines.append("## Filer-unit filter, target & negative rates")
lines.append("")
lines.append("- **Filer unit (§3.2, option A):** keep `depstat == 0` (non-dependent filers) -> 61,231 rows.")
lines.append("- **Target (§3.3):** `eff_rate = 100 x fedtaxac / adjginc`; all rows have `adjginc > 0`, so it is always defined.")
lines.append("- **Negative rates (§3.3):** ~12% of filer rows have `fedtaxac < 0` (refundable credits exceed tax owed). "
             "These negative effective rates are **kept, not clipped** — the net-progressive tail is a finding.")
lines.append("")

DICT_MD = ROOT / "data_dictionary.md"
DICT_MD.write_text("\n".join(lines) + "\n")
print("Wrote", DICT_MD.relative_to(ROOT), f"({len(lines)} lines)")
print("\n".join(lines[:22]))


Wrote data_dictionary.md (145 lines)
# Data Dictionary & Column Pile-Sort

**Source:** `data/raw/cps_filers_clean.csv` — 64,696 rows x 59 columns (IPUMS CPS ASEC 2024, income year 2023), one row per person.

> Auto-generated by `notebooks/build_modeling_table.ipynb` (Phase 0). Do not edit by hand — edit `COLUMN_PILES` in the notebook and re-run.

This is the **leakage firewall** required by the spec (README §3.4, §9 Phase 0): every column is assigned to exactly one pile. When column timing is ambiguous, it is quarantined.

## Pile summary

| Pile | Count | Meaning |
|---|---:|---|
| `target` | 1 | The value we predict. |
| `feature` | 15 | Pre-tax; enters the model matrix X. |
| `optional_feature` | 7 | Legitimate pre-tax axis, out of scope for the locked framing; documented, not in X. |
| `quarantine` | 9 | Target components or post-tax/credit outputs; never a feature (leakage). |
| `weight` | 2 | Survey weight; weighted validation only, never in X. |
| `sensitive` | 8 | Sensitive axi

## Phase 1 — Filer-unit filter, target confirmation & feature engineering

1. Filter to **filer units** — `depstat == 0` (non-dependent filers, §3.2 option A).
2. **Confirm the target** — `eff_rate == 100 x fedtaxac / adjginc`, always defined (`adjginc > 0`).
3. **Impute** `incretir` nulls to 0 (§3.4).
4. Build **income-composition shares** (guarded when `inctot <= 0`) — the core equity signal.
5. Assemble the modeling table `X + eff_rate + asecwt` and run a **leakage assertion**.


In [4]:
# --- 1. Filter to filer units (depstat == 0) --------------------------------
df = df_raw[df_raw["depstat"] == 0].copy()
print(f"Filer units (depstat==0): {len(df):,} rows  (spec expects 61,231)")
assert len(df) == 61231

# --- 2. Confirm the target --------------------------------------------------
assert (df["adjginc"] > 0).all(), "adjginc must be > 0 for eff_rate to be defined"
recomputed = 100 * df["fedtaxac"] / df["adjginc"]
max_diff = (df["eff_rate"] - recomputed).abs().max()
print(f"Target check: max |eff_rate - 100*fedtaxac/adjginc| = {max_diff:.2e}  (float noise)")
assert max_diff < 1e-6, "eff_rate does not match 100*fedtaxac/adjginc"

n_neg = int((df["fedtaxac"] < 0).sum())
print(f"eff_rate range: [{df['eff_rate'].min():.2f}, {df['eff_rate'].max():.2f}]  "
      f"(percent)")
print(f"Negative-rate filers (fedtaxac<0, refundable credits): {n_neg:,} "
      f"({100*n_neg/len(df):.1f}%) — KEPT, not clipped (§3.3)")

# --- 3. Impute incretir nulls to 0 ------------------------------------------
n_null_retir = int(df["incretir"].isna().sum())
df["incretir"] = df["incretir"].fillna(0)
print(f"Imputed incretir: {n_null_retir:,} nulls -> 0")


Filer units (depstat==0): 61,231 rows  (spec expects 61,231)
Target check: max |eff_rate - 100*fedtaxac/adjginc| = 1.42e-14  (float noise)
eff_rate range: [-49.99, 33.37]  (percent)
Negative-rate filers (fedtaxac<0, refundable credits): 7,344 (12.0%) — KEPT, not clipped (§3.3)
Imputed incretir: 42,778 nulls -> 0


In [5]:
# --- 4. Income-composition shares (guarded when inctot <= 0) ----------------
# Share = component / inctot when inctot > 0, else 0 (undefined/unstable base).
SHARE_SRC = {
    "wage_share":       "incwage",
    "business_share":   "incbus",
    "interest_share":   "incint",
    "dividend_share":   "incdivid",
    "retirement_share": "incretir",
    "socsec_share":     "incss",
    "rent_share":       "incrent",
}

pos = df["inctot"] > 0
n_guarded = int((~pos).sum())
for share, src in SHARE_SRC.items():
    df[share] = np.where(pos, df[src] / df["inctot"], 0.0)
print(f"Built {len(SHARE_SRC)} income shares; guarded (set to 0) where inctot<=0: {n_guarded:,} rows")
print(df[list(SHARE_SRC)].describe().round(3).T[["min", "50%", "max"]])

# --- 5. Assemble the modeling table (core features + target + weight) --------
NUMERIC_FEATURES = ["inctot", *SHARE_SRC.keys(), "age", "nchild", "nchlt5", "famsize"]
CATEGORICAL_FEATURES = ["filestat", "marst", "statefip"]  # keep as codes; encode at model time
FEATURE_COLS = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = "eff_rate"
WEIGHT = "asecwt"

model_df = df[FEATURE_COLS + [TARGET, WEIGHT]].copy()

# --- Leakage assertion: no feature is from a forbidden pile ------------------
FORBIDDEN = {"quarantine", "sensitive", "drop", "target", "weight"}
leaks = [c for c in FEATURE_COLS if pile_of.get(c) in FORBIDDEN]
assert not leaks, f"LEAKAGE: features drawn from forbidden piles: {leaks}"
# raw income components must NOT survive as columns (only their derived shares)
RAW_INCOME = ["incwage", "incbus", "incint", "incdivid", "incretir", "incss", "incrent"]
assert not (set(RAW_INCOME) & set(model_df.columns)), "raw income components leaked into table"
assert pile_of[TARGET] == "target" and pile_of[WEIGHT] == "weight"
assert model_df[FEATURE_COLS].isna().sum().sum() == 0, "unexpected nulls in features"
assert model_df[TARGET].isna().sum() == 0

print("\nLeakage assertion passed. Modeling table:", model_df.shape)
print("Features:", FEATURE_COLS)
print("Nulls in table:", int(model_df.isna().sum().sum()))


Built 7 income shares; guarded (set to 0) where inctot<=0: 1,179 rows
                     min    50%        max
wage_share         0.000  0.977  10000.000
business_share   -47.389  0.000      1.499
interest_share     0.000  0.000      5.286
dividend_share     0.000  0.000      1.000
retirement_share   0.000  0.000      1.000
socsec_share       0.000  0.000     11.159
rent_share       -10.162  0.000     23.697

Leakage assertion passed. Modeling table: (61231, 17)
Features: ['inctot', 'wage_share', 'business_share', 'interest_share', 'dividend_share', 'retirement_share', 'socsec_share', 'rent_share', 'age', 'nchild', 'nchlt5', 'famsize', 'filestat', 'marst', 'statefip']
Nulls in table: 0


In [6]:
# --- Persist the clean filer / modeling table (Phase 1 output) --------------
MODEL_CSV = PROCESSED / "cps_filers_modeling.csv"
MODEL_PARQUET = PROCESSED / "cps_filers_modeling.parquet"
model_df.to_csv(MODEL_CSV, index=False)
model_df.to_parquet(MODEL_PARQUET, index=False)
print("Wrote", MODEL_CSV.relative_to(ROOT), model_df.shape)
print("Wrote", MODEL_PARQUET.relative_to(ROOT))
model_df.head()


Wrote data/processed/cps_filers_modeling.csv (61231, 17)
Wrote data/processed/cps_filers_modeling.parquet


,inctot,wage_share,business_share,interest_share,dividend_share,retirement_share,socsec_share,rent_share,age,nchild,nchlt5,famsize,filestat,marst,statefip,eff_rate,asecwt
0,10801,0.00000,0.0,0.000185,0.000000,0.000000,0.999815,0.0,53,0,0,2,1,1,23,4.063351,1559.99
1,35987,0.00000,0.0,0.207714,0.005002,0.333454,0.453831,0.0,72,0,0,1,5,6,23,2.532584,1521.46
2,100,0.00000,0.0,1.000000,0.000000,0.000000,0.000000,0.0,62,1,0,3,1,1,23,5.104318,1500.23
3,54000,0.00000,0.0,0.000000,0.000000,0.000000,0.666667,0.0,58,0,0,2,1,1,23,0.000000,1351.18
4,43006,0.99986,0.0,0.000140,0.000000,0.000000,0.000000,0.0,57,1,0,3,1,1,23,6.465511,1351.18


## Phase 2 — Freeze the modeling table (blocking dependency)

Split the modeling table **80/20 with `random_state=42`** into `train.csv` /
`test.csv`. Per §3.5 this freeze is the **only blocking dependency** — once
frozen, no column may be added, and Phases 3-7 (RF, SHAP, twin, validation,
demo) run against these exact files. A `freeze_manifest.json` records the seed,
shapes, and content hashes so any later drift is detectable.


In [7]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    model_df, test_size=0.20, random_state=RANDOM_STATE, shuffle=True,
)
print(f"Split 80/20 (seed={RANDOM_STATE}):  train {train_df.shape}  test {test_df.shape}")
assert len(train_df) + len(test_df) == len(model_df)
assert set(train_df.index).isdisjoint(test_df.index), "train/test overlap!"

# --- Freeze verification: the split must be representative -------------------
def profile(name, d):
    neg = 100 * (d["eff_rate"] < 0).mean()
    return dict(split=name, rows=len(d),
                eff_mean=round(d["eff_rate"].mean(), 3),
                eff_std=round(d["eff_rate"].std(), 3),
                pct_negative=round(neg, 2),
                inctot_median=round(d["inctot"].median(), 1))

profile_tbl = pd.DataFrame([profile("full", model_df),
                            profile("train", train_df),
                            profile("test", test_df)]).set_index("split")
print("\nRepresentativeness (train/test should track full):")
print(profile_tbl.to_string())


Split 80/20 (seed=42):  train (48984, 17)  test (12247, 17)

Representativeness (train/test should track full):
        rows  eff_mean  eff_std  pct_negative  inctot_median
split                                                       
full   61231     5.352   10.216         11.99        50000.0
train  48984     5.310   10.286         12.21        50000.0
test   12247     5.521    9.929         11.15        50000.0


In [8]:
import hashlib, json

TRAIN_CSV = PROCESSED / "train.csv"
TEST_CSV  = PROCESSED / "test.csv"
train_df.to_csv(TRAIN_CSV, index=False)
test_df.to_csv(TEST_CSV, index=False)
# parquet copies for fast, type-stable downstream loads
train_df.to_parquet(PROCESSED / "train.parquet", index=False)
test_df.to_parquet(PROCESSED / "test.parquet", index=False)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

manifest = {
    "source": "data/processed/cps_filers_modeling.csv",
    "filer_unit_filter": "depstat == 0",
    "random_state": RANDOM_STATE,
    "test_size": 0.20,
    "shuffle": True,
    "target": TARGET,
    "weight_retained": WEIGHT,
    "feature_cols": FEATURE_COLS,
    "n_features": len(FEATURE_COLS),
    "rows": {"full": len(model_df), "train": len(train_df), "test": len(test_df)},
    "sha256": {"train.csv": sha256(TRAIN_CSV), "test.csv": sha256(TEST_CSV)},
    "note": "FROZEN modeling table (README Phase 2). Do not add columns; "
            "downstream phases 3-7 read these exact files.",
}
MANIFEST = PROCESSED / "freeze_manifest.json"
MANIFEST.write_text(json.dumps(manifest, indent=2) + "\n")

print("FROZEN:")
for p in (TRAIN_CSV, TEST_CSV, MANIFEST):
    print("  ", p.relative_to(ROOT))
print("\ntrain.csv sha256:", manifest["sha256"]["train.csv"][:16], "...")
print("test.csv  sha256:", manifest["sha256"]["test.csv"][:16], "...")


FROZEN:
   data/processed/train.csv
   data/processed/test.csv
   data/processed/freeze_manifest.json

train.csv sha256: 9cd1b12d19bf7b80 ...
test.csv  sha256: ab3e844cc3070861 ...


## Definition of Done — Phases 0-2

- [x] **Phase 0** — every one of the 59 columns assigned to exactly one pile,
      documented in `data_dictionary.md` (generated from `COLUMN_PILES`).
- [x] **Phase 1** — filer-unit table built (`depstat==0`, 61,231 rows);
      `eff_rate` target confirmed; income-composition shares engineered;
      leakage assertion passed.
- [x] **Phase 2** — modeling table **frozen** 80/20 (`random_state=42`) into
      `train.csv` / `test.csv`, with `freeze_manifest.json` (seed, shapes, hashes).

**Artifacts produced**

| Path | Phase |
|---|---|
| `data_dictionary.md` | 0 |
| `data/processed/cps_filers_modeling.{csv,parquet}` | 1 |
| `data/processed/train.{csv,parquet}`, `test.{csv,parquet}` | 2 |
| `data/processed/freeze_manifest.json` | 2 |

**Downstream (out of scope here):** Phase 3 Random Forest, Phase 4 SHAP,
Phase 5 twin comparison, Phase 6 IRS SOI validation, Phase 7 Streamlit demo.
Categorical features (`filestat`, `marst`, `statefip`) are kept as codes and
should be one-hot encoded at model time (Phase 3).
